In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp

## Introducción a CVXPY

`cvxpy` permite escribir un problema de optimización convexa casi igual que en el papel y resolverlo con un *solver* adecuado, sin programar el algoritmo de optimización. Ejecutá estas celdas en orden y modificalas para ver qué pasa.

Un problema se arma en cuatro pasos:

| Matemática | CVXPY |
|---|---|
| variable $x\in\mathbb R^2$ | `x = cp.Variable(2)` |
| $\text{minimizar}\ f(x)$ | `objetivo = cp.Minimize(f)` (o `cp.Maximize`) |
| sujeto a $g_i(x)\le h_i$ | `restricciones = [g @ x <= h, ...]` (una lista) |
| el problema | `problema = cp.Problem(objetivo, restricciones)` |

Después, `problema.solve()` lo resuelve y deja los resultados en `problema.status` (estado), `problema.value` (valor óptimo) y `x.value` (punto óptimo).

**Un primer ejemplo:**

$$
\begin{array}{ll}
\text{minimizar} & x_1+2x_2\\
\text{sujeto a} & x_1+x_2\ge1,\quad x_1\ge0,\quad x_2\ge0.
\end{array}
$$

In [ ]:
x = cp.Variable(2)  # variable de decisión, (2,)

objetivo = cp.Minimize(x[0] + 2 * x[1])
restricciones = [
    x[0] + x[1] >= 1,
    x >= 0,  # se aplica a cada componente
]
problema = cp.Problem(objetivo, restricciones)
problema.solve()

print("Estado:", problema.status)
print("Valor óptimo:", problema.value)
print("Punto óptimo:", x.value)

Las expresiones de `cvxpy` se escriben con la misma sintaxis que NumPy: `@` para el producto matricial, `<=`, `>=` y `==` para las restricciones (siempre entre expresiones, componente a componente). La misma restricción se puede escribir en forma matricial, $Gx\preceq h$:

In [ ]:
c = np.array([1.0, 2.0])
G = np.array([
    [-1.0, -1.0],  # -x1 - x2 <= -1, es decir x1 + x2 >= 1
    [-1.0, 0.0],  # -x1 <= 0
    [0.0, -1.0],  # -x2 <= 0
])
h = np.array([-1.0, 0.0, 0.0])

x = cp.Variable(2)
problema = cp.Problem(cp.Minimize(c @ x), [G @ x <= h])
problema.solve()
print(problema.status, problema.value, x.value)

**Revisar siempre el estado.** Si el problema no tiene solución, `x.value` queda en `None`. Los dos casos típicos:

- **Infactible**: las restricciones no se pueden cumplir a la vez (el valor óptimo es $+\infty$).
- **No acotado**: la función objetivo puede bajar indefinidamente dentro de la región factible (el valor óptimo es $-\infty$).

In [ ]:
# Infactible: x1 + x2 >= 1 y x1 + x2 <= 0.5, con x >= 0
x = cp.Variable(2)
problema = cp.Problem(
    cp.Minimize(x[0] + 2 * x[1]),
    [x[0] + x[1] >= 1, x[0] + x[1] <= 0.5, x >= 0],
)
problema.solve()
print("Estado:", problema.status, "| valor:", problema.value, "| x:", x.value)

# No acotado: se puede hacer x1 tan negativo como se quiera
problema = cp.Problem(cp.Minimize(x[0] + 2 * x[1]), [x[1] >= 0])
problema.solve()
print("Estado:", problema.status, "| valor:", problema.value, "| x:", x.value)

**Funciones de `cvxpy` que vamos a usar.** Reciben expresiones de `cvxpy` (no arrays de NumPy):

| Matemática | CVXPY |
|---|---|
| $\|v\|_2^2=\sum_iv_i^2$ | `cp.sum_squares(v)` |
| $\|v\|_1=\sum_i\lvert v_i\rvert$ | `cp.norm1(v)` |
| $\|v\|_\infty=\max_i\lvert v_i\rvert$ | `cp.norm_inf(v)` |
| $\|v\|_2$ | `cp.norm(v, 2)` |
| $x^\top Qx$ | `cp.quad_form(x, Q)` |
| $\sum_iv_i$ | `cp.sum(v)` |

In [ ]:
x = cp.Variable(2)
a = np.array([3.0, -1.0])

# Punto de la recta x1 + x2 = 1 más cercano a a, en norma 2 al cuadrado
problema = cp.Problem(cp.Minimize(cp.sum_squares(x - a)), [x[0] + x[1] == 1])
problema.solve()
print("Más cercano en norma 2:", x.value.round(4))

**CVXPY solo resuelve problemas convexos, y tiene que poder verificarlo.** Construye la función objetivo a partir de funciones cuya convexidad conoce, combinadas con reglas que preservan la convexidad (las de la guía 1.2). A esto le llama DCP (*disciplined convex programming*). Si no puede verificar que el problema es convexo, no lo resuelve. Por ejemplo, $x^\top Qx$ con $Q$ no semidefinida positiva no es convexa:

In [ ]:
Q_no_sdp = np.array([
    [1.0, 0.0],
    [0.0, -1.0],
])  # autovalores 1 y -1: indefinida

x = cp.Variable(2)
problema = cp.Problem(cp.Minimize(cp.quad_form(x, Q_no_sdp)), [cp.norm_inf(x) <= 1])
try:
    problema.solve()
except cp.error.DCPError:
    print("DCPError: el problema no cumple las reglas DCP (no es convexo)")

**Multiplicadores de Lagrange.** Después de resolver, cada restricción guarda su multiplicador óptimo en `restriccion.dual_value` (lo usamos en el ejercicio 1, ítem f):

In [ ]:
x = cp.Variable(2)
restriccion = x[0] + x[1] >= 1
problema = cp.Problem(cp.Minimize(cp.sum_squares(x)), [restriccion])
problema.solve()
print("x* =", x.value.round(4), "| multiplicador =", round(restriccion.dual_value, 4))

---

## Ejercicio 1

Problema en $\mathbb R^2$:

$$
\begin{array}{ll}
\text{minimizar} & \frac12 x^\top Qx+c^\top x\\
\text{sujeto a} & Gx\preceq h,
\end{array}
$$

con $G$ de forma `(m, 2)` (una fila por restricción) y $h$ de forma `(m,)`. Es un LP si $Q=0$ y un QP convexo si $Q\succeq0$.

Función auxiliar para graficar: pinta la región factible (los puntos de una grilla que cumplen $Gx\preceq h$), las curvas de nivel de la función objetivo, el punto óptimo y la curva de nivel que pasa por él.

In [ ]:
def graficar_problema(modelo, ax, lim=(-1, 5), titulo=""):
    """
    Grafica un problema de ProgramaCuadratico ya resuelto, en R^2.

    Parámetros:
    - modelo: ProgramaCuadratico ajustado
    - ax: eje de matplotlib
    - lim: límites de ambos ejes
    - titulo: título del gráfico
    """
    x1 = np.linspace(*lim, 400)
    xx1, xx2 = np.meshgrid(x1, x1)  # (400, 400)
    puntos = np.column_stack([xx1.ravel(), xx2.ravel()])  # (400*400, 2)

    # Región factible: todas las restricciones a la vez
    factible = np.all(puntos @ modelo.G.T <= modelo.h, axis=1).reshape(xx1.shape)
    ax.imshow(
        factible,
        extent=(*lim, *lim),
        origin="lower",
        cmap="Blues",
        alpha=0.4,
        vmin=0,
        vmax=1,
    )

    # Función objetivo en cada punto de la grilla
    F = puntos @ modelo.c
    if modelo.Q is not None:
        F = F + 0.5 * np.sum((puntos @ modelo.Q) * puntos, axis=1)
    F = F.reshape(xx1.shape)
    curvas = ax.contour(xx1, xx2, F, levels=15, colors="gray", linewidths=0.7)
    ax.clabel(curvas, fontsize=7)

    if modelo.x_opt is not None:
        ax.contour(xx1, xx2, F, levels=[modelo.valor_opt], colors="red", linewidths=1.5)
        ax.scatter(*modelo.x_opt, color="red", s=60, zorder=3, label=f"óptimo {np.round(modelo.x_opt, 2)}")
        ax.legend(loc="upper right", fontsize=8)

    ax.set_xlim(lim)
    ax.set_ylim(lim)
    ax.set_xlabel("$x_1$")
    ax.set_ylabel("$x_2$")
    ax.set_title(f"{titulo} ({modelo.estado})", fontsize=10)
    ax.set_aspect("equal")

#### a) Clase `ProgramaCuadratico`

In [ ]:
class ProgramaCuadratico:
    """
    Problema min 1/2 x^T Q x + c^T x  sujeto a  G x <= h, resuelto con cvxpy.
    """
    def __init__(self, Q=None):
        """
        Inicializa el problema.

        Parámetros:
        - Q: matriz (p, p) semidefinida positiva, o None para un LP
        """

        self.Q = Q

        # Resultados del ajuste
        self.c = None  # (p,)
        self.G = None  # (m, p)
        self.h = None  # (m,)
        self.x_opt = None  # punto óptimo (p,), None si no hay solución
        self.valor_opt = None  # valor óptimo
        self.estado = None  # "optimal", "infeasible", "unbounded", ...
        self.multiplicadores = None  # multiplicadores de Lagrange (m,)

    def fit(self, c, G, h):
        """
        Resuelve el problema.

        Parámetros:
        - c: (p,)
        - G: (m, p)
        - h: (m,)

        Retorna:
        - self
        """

        # TODO:
        # - Definir la variable x de dimensión p
        # - Armar la función objetivo: c^T x, más 1/2 x^T Q x si Q no es None
        # - Armar la restricción G x <= h y el problema; resolverlo
        # - Guardar c, G, h, el punto óptimo, el valor óptimo, el estado y los multiplicadores

        x = ...  # (p,)

        objetivo = ...
        if self.Q is not None:
            objetivo = ...

        restriccion = ...
        problema = ...
        problema.solve()

        self.c, self.G, self.h = c, G, h
        self.x_opt = ...  # (p,)
        self.valor_opt = ...
        self.estado = ...
        self.multiplicadores = ...  # (m,)

        return self

##### Solución

In [ ]:
class ProgramaCuadratico:
    """
    Problema min 1/2 x^T Q x + c^T x  sujeto a  G x <= h, resuelto con cvxpy.
    """
    def __init__(self, Q=None):
        """
        Inicializa el problema.

        Parámetros:
        - Q: matriz (p, p) semidefinida positiva, o None para un LP
        """

        self.Q = Q

        # Resultados del ajuste
        self.c = None  # (p,)
        self.G = None  # (m, p)
        self.h = None  # (m,)
        self.x_opt = None  # punto óptimo (p,), None si no hay solución
        self.valor_opt = None  # valor óptimo
        self.estado = None  # "optimal", "infeasible", "unbounded", ...
        self.multiplicadores = None  # multiplicadores de Lagrange (m,)

    def fit(self, c, G, h):
        """
        Resuelve el problema.

        Parámetros:
        - c: (p,)
        - G: (m, p)
        - h: (m,)

        Retorna:
        - self
        """

        x = cp.Variable(len(c))  # (p,)

        objetivo = c @ x
        if self.Q is not None:
            objetivo = 0.5 * cp.quad_form(x, self.Q) + c @ x

        restriccion = G @ x <= h
        problema = cp.Problem(cp.Minimize(objetivo), [restriccion])
        problema.solve()

        self.c, self.G, self.h = c, G, h
        self.x_opt = x.value  # (p,)
        self.valor_opt = problema.value
        self.estado = problema.status
        self.multiplicadores = restriccion.dual_value  # (m,)

        return self

#### b) El LP de la guía

$$
\text{maximizar}\ 2x_1+x_2\quad\text{sujeto a}\quad x_1+x_2\le4,\ 0\le x_1\le3,\ 0\le x_2\le3.
$$

Maximizar $2x_1+x_2$ es minimizar $-2x_1-x_2$, así que $c=(-2,-1)$. Cada desigualdad se escribe como una fila de $Gx\preceq h$ (las del tipo $x_1\ge0$ se pasan como $-x_1\le0$).

In [ ]:
c = np.array([-2.0, -1.0])
G = np.array([
    [1.0, 1.0],  # x1 + x2 <= 4
    [1.0, 0.0],  # x1 <= 3
    [0.0, 1.0],  # x2 <= 3
    [-1.0, 0.0],  # -x1 <= 0
    [0.0, -1.0],  # -x2 <= 0
])
h = np.array([4.0, 3.0, 3.0, 0.0, 0.0])

lp = ProgramaCuadratico().fit(c, G, h)
print(f"Estado: {lp.estado}")
print(f"x* = {lp.x_opt.round(4)}, valor óptimo del máximo = {-lp.valor_opt:.4f}")

fig, ax = plt.subplots(figsize=(5, 5))
graficar_problema(lp, ax, titulo="LP de la guía")
plt.show()

#### c) Infinitas soluciones, infactible y no acotado

In [ ]:
casos = {
    # c paralelo a la restricción x1 + x2 <= 4: todo un lado es óptimo
    "infinitas soluciones": (np.array([-1.0, -1.0]), G, h),
    # se agrega x1 + x2 >= 5, incompatible con x1 + x2 <= 4
    "infactible": (c, np.vstack([G, [-1.0, -1.0]]), np.append(h, -5.0)),
    # solo x >= 0: se puede crecer sin límite
    "no acotado": (c, G[3:], h[3:]),
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (nombre, (c_caso, G_caso, h_caso)) in zip(axes, casos.items()):
    modelo = ProgramaCuadratico().fit(c_caso, G_caso, h_caso)
    print(f"{nombre}: estado = {modelo.estado}, x* = {modelo.x_opt}, valor = {modelo.valor_opt}")
    graficar_problema(modelo, ax, titulo=nombre)
plt.tight_layout()
plt.show()

#### d) QP: óptimo en el interior y en el borde

Con $Q=I$, la función $\frac12\|x\|^2+c^\top x=\frac12\|x+c\|^2-\frac12\|c\|^2$ tiene curvas de nivel circulares centradas en $-c$, su mínimo sin restricciones. Si $-c$ es factible, es el óptimo; si no, el óptimo es el punto factible más cercano a $-c$, que está en el borde.

In [ ]:
Q = np.eye(2)
casos_qp = {
    "mínimo libre (1, 1), factible": np.array([-1.0, -1.0]),
    "mínimo libre (4, 4), no factible": np.array([-4.0, -4.0]),
}

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, (nombre, c_qp) in zip(axes, casos_qp.items()):
    qp = ProgramaCuadratico(Q).fit(c_qp, G, h)
    print(f"{nombre}: x* = {qp.x_opt.round(4)}, valor = {qp.valor_opt:.4f}")
    graficar_problema(qp, ax, titulo=nombre)
plt.tight_layout()
plt.show()

**Para discutir:** ¿Por qué en un QP el óptimo puede quedar en el interior de la región factible y en un LP no?

In [ ]:
try:
    ProgramaCuadratico(np.array([[1.0, 0.0], [0.0, -1.0]])).fit(c, G, h)
except cp.error.DCPError:
    print("DCPError: con Q indefinida el problema no es convexo")

#### e) *(Adicional)* Versión manual para el LP: enumeración de vértices

Cada vértice de la región factible es la intersección de dos restricciones activas: para cada par de filas $(i,j)$ de $G$ se resuelve el sistema $2\times2$

$$
\begin{pmatrix}G_i\\G_j\end{pmatrix}x=\begin{pmatrix}h_i\\h_j\end{pmatrix},
$$

se descartan los pares paralelos (determinante nulo) y los puntos no factibles, y se elige el vértice de menor $c^\top x$. Si el LP es factible y acotado, hay un vértice óptimo. Con `np.linalg.solve` se pueden resolver todos los sistemas a la vez, apilados en un array de forma `(P, 2, 2)`.

In [ ]:
from itertools import combinations


def resolver_lp_vertices(c, G, h, tol=1e-9):
    """
    Resuelve un LP en R^2 enumerando los vértices de la región factible.
    Supone que el LP es factible y acotado.

    Parámetros:
    - c: (2,)
    - G: (m, 2)
    - h: (m,)
    - tol: tolerancia para el determinante y la factibilidad

    Retorna:
    - x_opt: vértice óptimo (2,)
    - vertices: vértices factibles (V, 2)
    """

    # TODO:
    # - Armar los pares de índices de restricciones (P pares)
    # - Apilar los sistemas 2x2 y descartar los de determinante (casi) nulo
    # - Resolverlos todos juntos y quedarse con los vértices factibles
    # - Elegir el de menor c^T x

    pares = np.array(list(combinations(range(len(h)), 2)))  # (P, 2)
    A = ...  # (P, 2, 2)
    b = ...  # (P, 2)

    no_paralelas = ...  # (P,)
    # (P', 2, 2) con (P', 2, 1) -> (P', 2, 1) -> (P', 2)
    puntos = ...

    factibles = ...  # (P',)
    vertices = ...  # (V, 2)
    x_opt = ...  # (2,)

    return x_opt, vertices

##### Solución

In [ ]:
from itertools import combinations


def resolver_lp_vertices(c, G, h, tol=1e-9):
    """
    Resuelve un LP en R^2 enumerando los vértices de la región factible.
    Supone que el LP es factible y acotado.

    Parámetros:
    - c: (2,)
    - G: (m, 2)
    - h: (m,)
    - tol: tolerancia para el determinante y la factibilidad

    Retorna:
    - x_opt: vértice óptimo (2,)
    - vertices: vértices factibles (V, 2)
    """

    pares = np.array(list(combinations(range(len(h)), 2)))  # (P, 2)
    A = G[pares]  # (P, 2, 2)
    b = h[pares]  # (P, 2)

    no_paralelas = np.abs(np.linalg.det(A)) > tol  # (P,)
    # (P', 2, 2) con (P', 2, 1) -> (P', 2, 1) -> (P', 2)
    puntos = np.linalg.solve(A[no_paralelas], b[no_paralelas][:, :, None])[:, :, 0]

    factibles = np.all(puntos @ G.T <= h + tol, axis=1)  # (P',)
    vertices = puntos[factibles]  # (V, 2)
    x_opt = vertices[np.argmin(vertices @ c)]  # (2,)

    return x_opt, vertices

##### Verificación

In [ ]:
x_vert, vertices = resolver_lp_vertices(c, G, h)
print("Vértices factibles:\n", vertices.round(4))
print("Óptimo por vértices:", x_vert.round(4), "| con cvxpy:", lp.x_opt.round(4))
print("¿Coinciden?", np.allclose(x_vert, lp.x_opt, atol=1e-6))

#### f) *(Adicional)* Condiciones KKT y dualidad en el QP

Para el QP con el óptimo en el borde ($Q=I$, $c=(-4,-4)$), con $f(x)=\frac12x^\top Qx+c^\top x$ y multiplicadores $\lambda\in\mathbb R^m$, las condiciones KKT son:

- **Estacionariedad**: $\nabla f(x^\star)+G^\top\lambda^\star=Qx^\star+c+G^\top\lambda^\star=0$.
- **Factibilidad**: $Gx^\star\preceq h$ y $\lambda^\star\succeq0$.
- **Holgura complementaria**: $\lambda_i^\star\,(Gx^\star-h)_i=0$ para cada $i$: si una restricción no está activa, su multiplicador es 0.

La función dual es $g(\lambda)=\min_x\big[f(x)+\lambda^\top(Gx-h)\big]$. Igualando a cero el gradiente en $x$ queda $x=-Q^{-1}(c+G^\top\lambda)$, y reemplazando,

$$
g(\lambda)=-\tfrac12(c+G^\top\lambda)^\top Q^{-1}(c+G^\top\lambda)-\lambda^\top h.
$$

Si hay dualidad fuerte, $g(\lambda^\star)$ coincide con el valor óptimo del primal.

In [ ]:
Q = np.eye(2)
c_qp = np.array([-4.0, -4.0])
qp = ProgramaCuadratico(Q).fit(c_qp, G, h)
x_opt, lam = qp.x_opt, qp.multiplicadores

print("x* =", x_opt.round(4))
print("λ* =", lam.round(4))
print("Estacionariedad, Qx + c + G^T λ =", (Q @ x_opt + c_qp + G.T @ lam).round(6))
print("Factibilidad, G x - h =", (G @ x_opt - h).round(4), "| λ >= 0:", np.all(lam >= -1e-8))
print("Holgura complementaria, λ_i (G x - h)_i =", (lam * (G @ x_opt - h)).round(6))

z = c_qp + G.T @ lam  # (2,)
valor_dual = -0.5 * z @ np.linalg.solve(Q, z) - lam @ h
print(f"\nValor primal: {qp.valor_opt:.6f} | valor dual g(λ*): {valor_dual:.6f}")

---

## Ejercicio 2

Datos del modelo $Y=\beta_0+\beta_1X+\varepsilon$, con $X\sim\text{Unif}(-10,10)$ y $\varepsilon\sim\mathcal N(0,\sigma^2)$. Con la matriz de diseño $X_d$ de forma `(n, 2)` (una columna de unos y la de los $x_i$) y $\beta=(\beta_0,\beta_1)$, el vector de residuos es $r=y-X_d\beta$ y los tres criterios son $\min\|r\|_2^2$, $\min\|r\|_1$ y $\min\|r\|_\infty$.

In [ ]:
rng = np.random.default_rng(42)

n = 50
beta0, beta1 = 1.0, 2.0
sigma = 2.0

x = rng.uniform(-10, 10, n)  # (n,)
y = beta0 + beta1 * x + rng.normal(0, sigma, n)  # (n,)

fig, ax = plt.subplots(figsize=(7, 4))
ax.scatter(x, y, s=15)
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.show()

#### a) Clase `RegresionNorma`

In [ ]:
class RegresionNorma:
    """
    Recta y = beta0 + beta1 x ajustada minimizando una norma de los residuos, con cvxpy.
    """
    def __init__(self, norma="l2"):
        """
        Inicializa el modelo.

        Parámetros:
        - norma: "l2" (mínimos cuadrados), "l1" o "linf"
        """

        if norma not in ("l2", "l1", "linf"):
            raise ValueError(f"norma debe ser 'l2', 'l1' o 'linf', no {norma!r}")
        self.norma = norma

        # Resultados del ajuste
        self.beta = None  # (beta0, beta1), (2,)

    def matriz_diseno(self, x):
        """
        Matriz de diseño con una columna de unos.

        Parámetros:
        - x: (n,)

        Retorna:
        - X_d: (n, 2)
        """
        return np.column_stack([np.ones(len(x)), x])

    def fit(self, x, y):
        """
        Ajusta la recta.

        Parámetros:
        - x: (n,)
        - y: (n,)

        Retorna:
        - self
        """

        # TODO:
        # - Definir la variable beta y el vector de residuos
        # - Elegir la función objetivo según la norma (cp.sum_squares, cp.norm1, cp.norm_inf)
        # - Resolver y guardar beta

        X_d = self.matriz_diseno(x)  # (n, 2)
        beta = ...  # (2,)
        residuos = ...  # (n,)

        objetivos = {
            "l2": ...,
            "l1": ...,
            "linf": ...,
        }
        problema = ...
        problema.solve()

        self.beta = ...  # (2,)
        return self

    def predict(self, x):
        """
        Predicciones de la recta ajustada.

        Parámetros:
        - x: (n,)

        Retorna:
        - y_pred: (n,)
        """

        # TODO: y_pred = X_d beta

        y_pred = ...  # (n,)
        return y_pred

##### Solución

In [ ]:
class RegresionNorma:
    """
    Recta y = beta0 + beta1 x ajustada minimizando una norma de los residuos, con cvxpy.
    """
    def __init__(self, norma="l2"):
        """
        Inicializa el modelo.

        Parámetros:
        - norma: "l2" (mínimos cuadrados), "l1" o "linf"
        """

        if norma not in ("l2", "l1", "linf"):
            raise ValueError(f"norma debe ser 'l2', 'l1' o 'linf', no {norma!r}")
        self.norma = norma

        # Resultados del ajuste
        self.beta = None  # (beta0, beta1), (2,)

    def matriz_diseno(self, x):
        """
        Matriz de diseño con una columna de unos.

        Parámetros:
        - x: (n,)

        Retorna:
        - X_d: (n, 2)
        """
        return np.column_stack([np.ones(len(x)), x])

    def fit(self, x, y):
        """
        Ajusta la recta.

        Parámetros:
        - x: (n,)
        - y: (n,)

        Retorna:
        - self
        """

        X_d = self.matriz_diseno(x)  # (n, 2)
        beta = cp.Variable(2)  # (2,)
        residuos = y - X_d @ beta  # (n,)

        objetivos = {
            "l2": cp.sum_squares(residuos),
            "l1": cp.norm1(residuos),
            "linf": cp.norm_inf(residuos),
        }
        problema = cp.Problem(cp.Minimize(objetivos[self.norma]))
        problema.solve()

        self.beta = beta.value  # (2,)
        return self

    def predict(self, x):
        """
        Predicciones de la recta ajustada.

        Parámetros:
        - x: (n,)

        Retorna:
        - y_pred: (n,)
        """

        y_pred = self.matriz_diseno(x) @ self.beta  # (n,)
        return y_pred

##### Ajuste con las tres normas

In [ ]:
modelos = {norma: RegresionNorma(norma).fit(x, y) for norma in ("l2", "l1", "linf")}
for norma, modelo in modelos.items():
    print(f"{norma}: beta0 = {modelo.beta[0]:.4f}, beta1 = {modelo.beta[1]:.4f}")

#### b) Mínimos cuadrados con `np.linalg.lstsq`

In [ ]:
X_d = np.column_stack([np.ones(n), x])  # (n, 2)
beta_lstsq = np.linalg.lstsq(X_d, y, rcond=None)[0]  # (2,)

print("lstsq:", beta_lstsq.round(6))
print("cvxpy:", modelos["l2"].beta.round(6))
print("¿Coinciden?", np.allclose(beta_lstsq, modelos["l2"].beta, atol=1e-6))

#### c) Norma $\ell_1$ como LP

$|r_i|$ no es lineal, pero $t_i\ge|r_i|$ equivale a las dos desigualdades lineales $-t_i\le r_i\le t_i$. Con variables $(\beta,t)\in\mathbb R^{2+n}$:

$$
\begin{array}{ll}
\text{minimizar} & \sum_{i=1}^nt_i\\
\text{sujeto a} & -t\preceq y-X_d\beta\preceq t.
\end{array}
$$

En el óptimo cada $t_i$ baja hasta $|r_i|$, porque si no se podría achicar y bajar el objetivo: el LP y el problema original tienen la misma solución $\beta$.

In [ ]:
def ajuste_l1_lp(x, y):
    """
    Ajuste en norma l1 formulado como LP con variables auxiliares t_i >= |r_i|.

    Parámetros:
    - x: (n,)
    - y: (n,)

    Retorna:
    - beta: (2,)
    """

    # TODO:
    # - Variables beta (2,) y t (n,)
    # - Restricciones -t <= y - X_d beta <= t
    # - Minimizar la suma de t

    X_d = np.column_stack([np.ones(len(x)), x])  # (n, 2)
    beta = ...  # (2,)
    t = ...  # (n,)
    residuos = ...  # (n,)

    restricciones = [
        ...,
        ...,
    ]
    problema = ...
    problema.solve()

    return beta.value

##### Solución

In [ ]:
def ajuste_l1_lp(x, y):
    """
    Ajuste en norma l1 formulado como LP con variables auxiliares t_i >= |r_i|.

    Parámetros:
    - x: (n,)
    - y: (n,)

    Retorna:
    - beta: (2,)
    """

    X_d = np.column_stack([np.ones(len(x)), x])  # (n, 2)
    beta = cp.Variable(2)  # (2,)
    t = cp.Variable(len(x))  # (n,)
    residuos = y - X_d @ beta  # (n,)

    restricciones = [
        residuos <= t,
        -t <= residuos,
    ]
    problema = cp.Problem(cp.Minimize(cp.sum(t)), restricciones)
    problema.solve()

    return beta.value

##### Verificación

In [ ]:
beta_l1_lp = ajuste_l1_lp(x, y)
print("LP explícito:", beta_l1_lp.round(6))
print("Forma directa:", modelos["l1"].beta.round(6))
print("¿Coinciden?", np.allclose(beta_l1_lp, modelos["l1"].beta, atol=1e-5))

#### d) Rectas ajustadas, sin y con outliers

In [ ]:
def graficar_ajustes(x, y, modelos, ax, titulo=""):
    """
    Datos y rectas ajustadas por cada norma.

    Parámetros:
    - x, y: datos (n,)
    - modelos: diccionario {norma: RegresionNorma ajustada}
    - ax: eje de matplotlib
    """
    ax.scatter(x, y, s=15, color="gray")
    x_grilla = np.linspace(x.min(), x.max(), 100)
    estilos = {"l2": "k--", "l1": "g-", "linf": "r-"}
    for norma, modelo in modelos.items():
        b0, b1 = modelo.beta
        ax.plot(
            x_grilla,
            modelo.predict(x_grilla),
            estilos[norma],
            label=f"{norma}: {b0:.2f} + {b1:.2f} x",
        )
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(titulo)
    ax.legend(fontsize=8)


# Outliers: 5 observaciones con y muy por encima de la recta
indices = rng.choice(n, 5, replace=False)
y_out = y.copy()
y_out[indices] += rng.uniform(20, 40, 5)

modelos_out = {norma: RegresionNorma(norma).fit(x, y_out) for norma in ("l2", "l1", "linf")}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
graficar_ajustes(x, y, modelos, axes[0], "Sin outliers")
graficar_ajustes(x, y_out, modelos_out, axes[1], "Con 5 outliers")
axes[1].scatter(x[indices], y_out[indices], s=60, facecolors="none", edgecolors="r")
plt.tight_layout()
plt.show()

print("Verdaderos: beta0 = 1, beta1 = 2")
for norma in modelos:
    print(
        f"{norma}: sin outliers {modelos[norma].beta.round(3)}, "
        f"con outliers {modelos_out[norma].beta.round(3)}"
    )

**Para discutir:** ¿Qué ajuste se ve más afectado por los outliers? ¿Por qué?

#### e) *(Adicional)* $\ell_\infty$ como LP y $\ell_2$ como QP

- $\ell_\infty$: una sola variable auxiliar $s\ge|r_i|$ para todo $i$: minimizar $s$ sujeto a $-s\mathbf 1\preceq y-X_d\beta\preceq s\mathbf 1$.
- $\ell_2$: desarrollando el cuadrado, $\|y-X_d\beta\|^2=\beta^\top X_d^\top X_d\beta-2y^\top X_d\beta+y^\top y$, que es un QP $\frac12\beta^\top Q\beta+q^\top\beta$ con $Q=2X_d^\top X_d$ y $q=-2X_d^\top y$ (la constante $y^\top y$ no cambia el minimizador).

In [ ]:
def ajuste_linf_lp(x, y):
    """
    Ajuste en norma l-infinito formulado como LP con una variable auxiliar s >= |r_i|.

    Parámetros:
    - x: (n,)
    - y: (n,)

    Retorna:
    - beta: (2,)
    """

    # TODO: variables beta (2,) y s (escalar); -s <= r_i <= s; minimizar s

    X_d = np.column_stack([np.ones(len(x)), x])  # (n, 2)
    beta = ...  # (2,)
    s = ...  # escalar
    residuos = ...  # (n,)
    problema = ...
    problema.solve()

    return beta.value


def ajuste_l2_qp(x, y):
    """
    Ajuste en norma l2 formulado como QP: 1/2 beta^T Q beta + q^T beta.

    Parámetros:
    - x: (n,)
    - y: (n,)

    Retorna:
    - beta: (2,)
    """

    # TODO: Q = 2 X_d^T X_d, q = -2 X_d^T y; minimizar 1/2 beta^T Q beta + q^T beta

    X_d = np.column_stack([np.ones(len(x)), x])  # (n, 2)
    Q = ...  # (2, 2)
    q = ...  # (2,)
    beta = ...  # (2,)
    problema = ...
    problema.solve()

    return beta.value

##### Solución

In [ ]:
def ajuste_linf_lp(x, y):
    """
    Ajuste en norma l-infinito formulado como LP con una variable auxiliar s >= |r_i|.

    Parámetros:
    - x: (n,)
    - y: (n,)

    Retorna:
    - beta: (2,)
    """

    X_d = np.column_stack([np.ones(len(x)), x])  # (n, 2)
    beta = cp.Variable(2)  # (2,)
    s = cp.Variable()  # escalar
    residuos = y - X_d @ beta  # (n,)
    problema = cp.Problem(cp.Minimize(s), [residuos <= s, -s <= residuos])
    problema.solve()

    return beta.value


def ajuste_l2_qp(x, y):
    """
    Ajuste en norma l2 formulado como QP: 1/2 beta^T Q beta + q^T beta.

    Parámetros:
    - x: (n,)
    - y: (n,)

    Retorna:
    - beta: (2,)
    """

    X_d = np.column_stack([np.ones(len(x)), x])  # (n, 2)
    Q = 2 * X_d.T @ X_d  # (2, 2)
    q = -2 * X_d.T @ y  # (2,)
    beta = cp.Variable(2)  # (2,)
    problema = cp.Problem(cp.Minimize(0.5 * cp.quad_form(beta, Q) + q @ beta))
    problema.solve()

    return beta.value

##### Verificación

In [ ]:
print("linf, LP explícito:", ajuste_linf_lp(x, y).round(5), "| forma directa:", modelos["linf"].beta.round(5))
print("l2, QP explícito:", ajuste_l2_qp(x, y).round(5), "| forma directa:", modelos["l2"].beta.round(5))